# ShopAgent-AI | CP4: Cart, Order, and Invoice

**Objective:** Enable the agent to **execute real actions**: adding items to the cart, placing an order, deducting inventory, and generating a PDF invoice.

**Design Decisions:**
1. **Database-backed Cart:** The cart is stored in the database (a `cart_items` table linked to the `thread_id`) rather than the LLM's memory, to avoid issues like forgetting or hallucinations.
2. **Atomic Order Transaction:** The order process runs as a single transaction: checking inventory, recording the order, deducting stock, and clearing the cart. It is an all-or-nothing operation.
3. **Confirmation Logic:** The `create_order` tool rejects requests unless the customer's last message contains an explicit confirmation keyword (verified by code, not the model). This prevents the LLM from unilaterally deciding to make a purchase.
4. **Router:** Directs all purchase and cart-related operations to the large model.
> **Limitations of this confirmation:** It relies on keyword checking; if the customer types "yes" while intending something else, it might still pass. The complete solution in CP5: The **Confirm** button in the interface is what actually executes the order.

**CP4 Success Criteria:**
- [ ] All tests in `test_orders.py` pass
- [ ] Complete workflow: Search → Cart → Confirm → Order → Invoice
- [ ] Inventory is actually deducted in the database
- [ ] A purchase request without explicit confirmation is rejected

## 1. Settings

In [1]:
import os
import shutil
import time
import uuid
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))

GROQ_FAST_MODEL = os.getenv("GROQ_FAST_MODEL", "openai/gpt-oss-20b")   
GROQ_SMART_MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-120b")      
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "qwen2.5:3b")                

DEMO_CUSTOMER_ID = 1 

print("GROQ_API_KEY set:", bool(os.getenv("GROQ_API_KEY")))
print("Models:", GROQ_FAST_MODEL, "|", GROQ_SMART_MODEL, "|", OLLAMA_MODEL)

GROQ_API_KEY set: True
Models: openai/gpt-oss-20b | openai/gpt-oss-120b | qwen2.5:3b


## 2. Database: Backup and Cart Table
The CP4 Agent **actually modifies the data** (deducting inventory and recording orders). Therefore:
- We create a **backup** of `shop.db` once.
- The `reset_demo()` function restores the original inventory and clears orders and carts, allowing you to restart the demo from scratch for the team.

In [2]:
import db
import orders

# One-time safety copy, made before the agent starts changing stock
backup = Path(db.DB_PATH).with_name("shop_backup.db")
if not backup.exists():
    shutil.copy(db.DB_PATH, backup)
    print("Backup created:", backup)

orders.init_orders_schema()


def reset_demo():
    """Restore the original stock and delete all carts and orders."""
    with db.get_conn() as conn:
        conn.execute("ATTACH DATABASE ? AS bk", (str(backup),))
        conn.execute(
            "UPDATE products SET stock = (SELECT b.stock FROM bk.products b WHERE b.id = products.id)"
        )
        conn.execute("DELETE FROM order_items")
        conn.execute("DELETE FROM orders")
        conn.execute("DELETE FROM cart_items")
        conn.commit()
        conn.execute("DETACH DATABASE bk")
    print("Demo reset: stock restored, carts and orders cleared")


print("Database ready:", db.DB_PATH)

Database ready: data/shop.db


## 3. Tools
- **CP3 Tools** (search, details, comparison, inventory): Remain unchanged.
- **CP4 Tools:** `add_to_cart`, `remove_from_cart`, `view_cart`, `create_order`, and `generate_invoice`.

**Two important points:**
- The tools utilize a `thread_id` from the `config` object, which is automatically injected and hidden from the LLM; consequently, the LLM cannot alter it or access another user's cart.
- The `create_order` tool takes no parameters; instead, it reads the **confirmation flag** from the `config` object—a value set by the `chat()` function (see Section 7). The LLM cannot forge this flag.

In [3]:
from langchain_core.runnables import RunnableConfig
from langchain_core.tools import tool

import tools as store  # CP2 module (tools.py)


def shorten(text, limit=80):
    """Cut long text so tool results stay small for the LLM."""
    text = str(text)
    return text if len(text) <= limit else text[: limit - 3] + "..."


def short_names(rows):
    return [{**row, "name": shorten(row["name"])} for row in rows]


def short_cart(result):
    """Shorten product names inside a cart (or inside a result that contains one)."""
    cart = result.get("cart", result)
    if "items" in cart:
        cart["items"] = short_names(cart["items"])
    return result


# ---------------- CP3 tools: browsing ----------------
@tool
def search_products(
    query: str,
    max_price: float | None = None,
    min_rating: float | None = None,
    category: str | None = None,
) -> dict:
    """Search store products by keyword. Product names are in English, so translate
    the customer's request to ONE or TWO short English keywords first (for example
    'cable', 'charger', 'headphones'). Only use max_price, min_rating or category
    if the customer explicitly asked for them. Prices are in INR, rating is 0-5.
    Returns up to 5 products with id, name, price, stock, rating."""
    if not query.strip():
        return {"count": 0, "products": [], "message": "Empty search. Provide a keyword."}
    products = short_names(store.search_products(query, max_price, min_rating, category))
    if not products:
        # Never return an empty list: some LLM APIs reject empty tool results
        return {
            "count": 0,
            "products": [],
            "message": "No products found. Try a shorter English keyword or remove the filters.",
        }
    return {"count": len(products), "products": products}


@tool
def get_product_details(product_id: int) -> dict:
    """Get full details (brand, price, stock, rating, description) of one product by its id."""
    details = store.get_product_details(product_id)
    if "error" not in details:
        details["name"] = shorten(details["name"])
        details["description"] = shorten(details.get("description") or "", 300)
    return details


@tool
def compare_products(product_ids: list[int]) -> dict:
    """Compare 2 to 4 products by id on price, rating and stock.
    Also tells which one is the cheapest and which one is the best rated."""
    result = store.compare_products(product_ids)
    if "products" in result:
        result["products"] = short_names(result["products"])
    return result


@tool
def check_stock(product_id: int, quantity: int = 1) -> dict:
    """Check if the requested quantity of a product (by id) is available in stock."""
    result = store.check_stock(product_id, quantity)
    if "name" in result:
        result["name"] = shorten(result["name"])
    return result


# ---------------- CP4 tools: cart, order, invoice ----------------
def _thread(config):
    return config["configurable"]["thread_id"]


@tool
def add_to_cart(product_id: int, quantity: int = 1, config: RunnableConfig = None) -> dict:
    """Add a product (by id) to the customer's cart. Use it only after the customer
    chose the product. If they did not give a quantity, use 1. Refuses quantities
    above the available stock."""
    return short_cart(orders.add_to_cart(_thread(config), product_id, quantity))


@tool
def remove_from_cart(product_id: int, quantity: int | None = None, config: RunnableConfig = None) -> dict:
    """Remove a product (by id) from the cart. Give a quantity to remove only some
    units, or leave it empty to remove the product completely."""
    return short_cart(orders.remove_from_cart(_thread(config), product_id, quantity))


@tool
def view_cart(config: RunnableConfig = None) -> dict:
    """Show the customer's cart: items, quantities, line totals and the subtotal."""
    return short_cart(orders.view_cart(_thread(config)))


@tool
def create_order(config: RunnableConfig = None) -> dict:
    """Place the order for everything in the cart and deduct the stock. Call it ONLY
    after you showed the cart summary and the customer's latest message clearly
    confirmed the order. The system checks the confirmation itself."""
    cfg = config["configurable"]
    if not cfg.get("order_confirmed"):
        return {
            "error": "The customer has not confirmed in their last message. "
                     "Show the cart summary and ask them to confirm explicitly.",
            "cart": short_cart(orders.view_cart(_thread(config)))["items"] or "empty",
        }
    return orders.create_order(_thread(config), cfg.get("customer_id", DEMO_CUSTOMER_ID))


@tool
def generate_invoice(order_id: int) -> dict:
    """Create a PDF invoice for an order id. Returns the file path of the invoice."""
    return orders.generate_invoice(order_id)


TOOLS = [
    search_products, get_product_details, compare_products, check_stock,
    add_to_cart, remove_from_cart, view_cart, create_order, generate_invoice,
]
print("Tools ready:", [t.name for t in TOOLS])

Tools ready: ['search_products', 'get_product_details', 'compare_products', 'check_stock', 'add_to_cart', 'remove_from_cart', 'view_cart', 'create_order', 'generate_invoice']


## 4. building models


In [4]:
def build_llms():
    """Create every model that can be created. A missing key just skips that model."""
    llms = {}

    try:
        from langchain_groq import ChatGroq

        llms["groq-fast"] = ChatGroq(model=GROQ_FAST_MODEL, temperature=0)
        llms["groq-smart"] = ChatGroq(model=GROQ_SMART_MODEL, temperature=0)
    except Exception as exc:
        print(f"[warn] groq not available: {exc}")

    try:
        from langchain_ollama import ChatOllama

        llms["ollama"] = ChatOllama(model=OLLAMA_MODEL, temperature=0)
    except Exception as exc:
        print(f"[warn] ollama not available: {exc}")

    return list(llms.items())


LLMS = build_llms()
print("Models created:", [name for name, _ in LLMS])

Models created: ['groq-fast', 'groq-smart', 'ollama']


## 5. Health check

In [5]:
def check_models(llms):
    for name, llm in llms:
        start = time.time()
        try:
            llm.invoke("Reply with the single word OK")
            print(f"[OK]   {name:<10} {time.time() - start:5.1f}s")
        except Exception as exc:
            print(f"[FAIL] {name:<10} {type(exc).__name__}: {str(exc)[:150]}")


check_models(LLMS)

[OK]   groq-fast    2.1s
[OK]   groq-smart   3.7s
[OK]   ollama      22.3s


## 6. Agents and Memory
The system prompt has been expanded with purchasing rules: it displays a cart summary before placing the order, only confirms the order has been placed if `create_order` returns an `order_id`, and automatically generates the invoice upon successful order completion.

In [6]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent

SYSTEM_PROMPT = """You are ShopAgent, a helpful assistant for an online store.

Browsing rules:
- Always use the tools to look up products. Never invent products, prices or stock.
- Product names in the database are in English. Translate the customer's request
  to one or two short English keywords before searching, but reply in the
  customer's language (Arabic or English).
- When you list products, show the id, a short name, the price and the rating.
- When the customer says "the first", "the second" or "the last one", refer to the
  products you listed in your previous message and use their ids.
- Prices are in Indian rupees (INR) unless told otherwise.

Cart and order rules:
- Use add_to_cart only after the customer chose a product. If no quantity is given, use 1.
- Before placing an order, call view_cart, show the items and the total, and ask the
  customer to confirm. Never call create_order before that.
- Call create_order only when the customer's latest message clearly confirms the
  order. The system verifies this itself; if create_order returns an error, tell
  the customer what is needed.
- Never say an order was placed unless create_order returned an order_id.
- After create_order succeeds, call generate_invoice with that order_id, then tell
  the customer the order id, the total and the invoice file path.
- If a tool returns an error, explain it honestly and suggest a fix (for example
  a smaller quantity).
- If the question is not about the store, politely say you can only help with shopping.
"""

MEMORY = MemorySaver()  # one memory shared by all agents

AGENTS = [
    (name, create_react_agent(llm, TOOLS, checkpointer=MEMORY, prompt=SYSTEM_PROMPT))
    for name, llm in LLMS
]
print("Agents ready:", [name for name, _ in AGENTS])

Agents ready: ['groq-fast', 'groq-smart', 'ollama']


C:\Users\GPU-TECH\AppData\Local\Temp\ipykernel_1568\2037095335.py:34: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  (name, create_react_agent(llm, TOOLS, checkpointer=MEMORY, prompt=SYSTEM_PROMPT))


## 7. The Router, Confirmation, and the `chat()` Function

In [7]:
import re


def as_text(content):
    """Some models return a list of content blocks instead of a plain string."""
    if isinstance(content, list):
        return "".join(
            part.get("text", "") if isinstance(part, dict) else str(part) for part in content
        )
    return str(content)


# ---------------- Text helpers ----------------
def norm(text):
    """Lowercase, drop Arabic diacritics and unify letter variants for forgiving matching."""
    text = re.sub(r"[\u064B-\u0652]", "", text.lower())
    for a, b in (("أ", "ا"), ("إ", "ا"), ("آ", "ا"), ("ى", "ي"), ("ة", "ه")):
        text = text.replace(a, b)
    return text


def has_any(text, words):
    tokens = set(re.findall(r"\w+", text))
    return any((w in text) if (" " in w or len(w) >= 4) else (w in tokens) for w in words)


# Words that signal a task needing more reasoning
COMPLEX_WORDS = [norm(w) for w in (
    "قارن", "مقارنه", "الفرق", "الافضل", "افضل", "احسن", "رشح", "رشحلي", "انصحني",
    "compare", "difference", "versus", "vs", "best", "recommend", "better", "which one",
)]
# Words that point back to earlier results (the model must use the conversation memory)
REFERENCE_WORDS = [norm(w) for w in (
    "الاول", "التاني", "الثاني", "التالت", "الثالث", "الاخير", "ده", "دي",
    "first", "second", "third", "last one", "that one", "previous",
)]
# Words about changing state (cart, order, invoice): these need the stronger model
STATE_WORDS = [norm(w) for w in (
    "سله", "السله", "اشتر", "اطلب", "اكد", "تاكيد", "احذف", "شيل", "فاتوره",
    "cart", "buy", "order", "confirm", "checkout", "remove", "invoice", "add",
)]

CONFIRM_WORDS = [norm(w) for w in (
    "اكد", "اكده", "تاكيد", "confirm", "yes", "ايوه", "اه", "موافق", "نعم",
    "go ahead", "place the order", "place order",
)]
NEGATION_WORDS = [norm(w) for w in (
    "لا", "مش", "ما", "no", "not", "dont", "don't", "cancel", "الغي", "الغاء",
)]


def is_confirmation(message):
    """True only if the message explicitly confirms (and does not negate)."""
    text = norm(message)
    return has_any(text, CONFIRM_WORDS) and not has_any(text, NEGATION_WORDS)


# ---------------- Router ----------------
def classify(message):
    """Return 'complex' or 'simple' using cheap keyword rules (no LLM call)."""
    text = norm(message)
    if has_any(text, COMPLEX_WORDS) or has_any(text, REFERENCE_WORDS) or has_any(text, STATE_WORDS):
        return "complex"
    if len(text) > 120 or len(re.findall(r"\d+", text)) >= 2:
        return "complex"
    return "simple"


ROUTES = {
    "complex": ["groq-smart", "groq-fast", "ollama"],
    "simple": ["groq-fast", "groq-smart", "ollama"],
}


def route(message):
    """Return (task_kind, ordered list of available models). Ollama is always last."""
    kind = classify(message)
    available = {name for name, _ in AGENTS}
    return kind, [name for name in ROUTES[kind] if name in available]


# ---------------- Chat with fallback ----------------
def chat(message, thread_id="demo", skip=()):
    config = {
        "configurable": {
            "thread_id": thread_id,
            "customer_id": DEMO_CUSTOMER_ID,
            "order_confirmed": is_confirmation(message),  
        }
    }
    kind, order = route(message)
    agents = dict(AGENTS)
    errors = []

    for name in order:
        if name in skip:
            continue
        start = time.time()
        try:
            result = agents[name].invoke({"messages": [("user", message)]}, config)
        except Exception as exc:  # connection, rate limit, wrong key, ...
            errors.append(f"{name}: {type(exc).__name__}: {str(exc)[:150]}")
            print(f"[warn] {name} failed -> trying the next model")
            continue

        messages = result["messages"]
        last_human = max(i for i, m in enumerate(messages) if m.type == "human")
        tools_used = [
            call["name"]
            for m in messages[last_human + 1 :]
            for call in (getattr(m, "tool_calls", None) or [])
        ]
        return {
            "reply": as_text(messages[-1].content),
            "model": name,
            "task": kind,
            "seconds": round(time.time() - start, 1),
            "tools": tools_used,
        }

    raise RuntimeError("All models failed:\n" + "\n".join(errors))


def show(result):
    print(f"[{result['task']} -> {result['model']} | {result['seconds']}s | tools: {result['tools']}]")
    print(result["reply"])

### 7.1 Router Testing and Verification (Without Any API Calls)

In [8]:
for text in [
    "عاوز كابل شحن",
    "ضيف الأول للسلة",
    "أكّد الطلب",
    "Add product 1512 to my cart",
    "yes, confirm my order",
    "لا مش عايز أكد",
    "Order it now",
    "I want a charging cable",
]:
    kind, order = route(text)
    print(f"{kind:<8} -> {order[0]:<10} confirm={is_confirmation(text)!s:<5} | {text}")

simple   -> groq-fast  confirm=False | عاوز كابل شحن
complex  -> groq-smart confirm=False | ضيف الأول للسلة
complex  -> groq-smart confirm=True  | أكّد الطلب
complex  -> groq-smart confirm=False | Add product 1512 to my cart
complex  -> groq-smart confirm=True  | yes, confirm my order
complex  -> groq-smart confirm=False | لا مش عايز أكد
complex  -> groq-smart confirm=False | Order it now
simple   -> groq-fast  confirm=False | I want a charging cable


## 8. The Full Workflow: From Search to Invoice
The action is actually executed on the database, so the inventory level will decrease. You can revert this at any time using `reset_demo()`.

In [9]:
def stock_of(product_id):
    with db.get_conn() as conn:
        return conn.execute("SELECT stock FROM products WHERE id = ?", (product_id,)).fetchone()["stock"]


thread = str(uuid.uuid4())
tracked_id = before_stock = None

for i, question in enumerate([
    "عاوز كابل شحن تحت 500",
    "ضيف الأول للسلة، 2 قطعة",
    "إيه اللي في السلة؟",
    "أكّد الطلب",
]):
    print(f"\nYou: {question}")
    show(chat(question, thread))

    if i == 1:  # right after "add to cart": remember the product and its stock
        cart = orders.view_cart(thread)
        if cart["items"]:
            tracked_id = cart["items"][0]["product_id"]
            before_stock = stock_of(tracked_id)

if tracked_id:
    print(f"\nStock of product {tracked_id}: before={before_stock}, after={stock_of(tracked_id)}")


You: عاوز كابل شحن تحت 500
[simple -> groq-fast | 3.8s | tools: ['search_products']]
إليك بعض كابلات الشحن التي تتوفر بأسعار أقل من 500 روبية:

| # | معرف المنتج | الاسم المختصر | السعر (₹) | التقييم |
|---|-------------|---------------|-----------|---------|
| 1 | **1512** | Syncwire LTG USB Cable | 399 | 5.0 |
| 2 | **1660** | REDTECH USB‑C to Lightning Cable | 249 | 5.0 |
| 3 | **1408** | Oraimo 65W USB‑C to USB‑C Cable | 349 | 4.5 |
| 4 | **1426** | Lapster 65W USB‑C to USB‑C Cable | 199 | 4.5 |
| 5 | **1850** | Noise Colorfit Smart Watch Charger | 249 | 4.5 |

أيّ كابل ترغب في إضافته إلى سلة التسوق؟ إذا كنت تريد كابلًا محددًا، أخبرني برقم المعرف أو بالاسم المختصر.

You: ضيف الأول للسلة، 2 قطعة
[complex -> groq-smart | 2.4s | tools: ['check_stock', 'add_to_cart']]
تم إضافة 2 قطعة من **Syncwire LTG to USB Cable** إلى سلة التسوق.

**ملخص السلة**

| المعرف | الاسم | السعر (₹) | الكمية | المجموع (₹) |
|--------|-------|-----------|--------|-------------|
| 1512 | Syncwire LTG to USB C

## 9. Database Verification
An agent might report "Done" without actually having performed the task; the true proof lies in the database—specifically, the latest order and its line items, as well as the generated invoices.

In [10]:
def show_last_order():
    with db.get_conn() as conn:
        order = conn.execute("SELECT * FROM orders ORDER BY id DESC LIMIT 1").fetchone()
        if order is None:
            print("No orders yet.")
            return
        print("Last order:", dict(order))
        for row in conn.execute(
            "SELECT p.name, oi.quantity, oi.unit_price FROM order_items oi "
            "JOIN products p ON p.id = oi.product_id WHERE oi.order_id = ?",
            (order["id"],),
        ):
            print("  item:", shorten(row["name"], 50), "| qty", row["quantity"], "| price", row["unit_price"])
    print("Invoices:", sorted(p.name for p in Path(orders.INVOICE_DIR).glob("*.pdf")))


show_last_order()

Last order: {'id': 2, 'customer_id': 1, 'total': 798.0, 'status': 'confirmed', 'created_at': '2026-10-10 09:48:12'}
  item: Syncwire LTG to USB Cable for Fast Charging Com... | qty 2 | price 399.0
Invoices: ['invoice_1_20261010_124129.pdf', 'invoice_2_20261010_124825.pdf', 'invoice_3_20261010_124554.pdf']


## 10. Security Test: Unconfirmed Purchase Order
We place a product in the cart and instruct the agent to "order" it without a confirmation step. Even if the LLM attempts to call `create_order`, the code must reject the request. We then verify the database to ensure the order count and inventory levels remain unchanged.

In [ ]:
candidates = [p for p in store.search_products("cable") if p["stock"] >= 1]
guard_thread = str(uuid.uuid4())
pid = candidates[0]["id"]
orders.add_to_cart(guard_thread, pid, 1)

with db.get_conn() as conn:
    orders_before = conn.execute("SELECT COUNT(*) AS n FROM orders").fetchone()["n"]
stock_before = stock_of(pid)

show(chat("Order it now", guard_thread)) 

with db.get_conn() as conn:
    orders_after = conn.execute("SELECT COUNT(*) AS n FROM orders").fetchone()["n"]

ok = orders_after == orders_before and stock_of(pid) == stock_before
print("\nGUARD", "PASS" if ok else "FAIL", f"| orders {orders_before}->{orders_after} | stock {stock_before}->{stock_of(pid)}")
orders.clear_cart(guard_thread)

[complex -> groq-smart | 4.6s | tools: []]
أعتذر، لكن لم يتم تحديد أي منتج تريد طلبه. من فضلك أخبرني باسم أو نوع المنتج الذي ترغب في شرائه (مثلاً : هاتف ، سماعات ، شاحن ) أو أرسل لي تفاصيل المنتج إذا كان لديك. سأسعد بمساعدتك في إتمام الطلب!

GUARD PASS | orders 2->2 | stock 37->37


## 11. Mini-Evaluation
Same concept as CP3: we test with questions and check whether the expected tool was called. Each question takes place in a new conversation, and the context is cleared afterward.
> This consumes API requests (7 questions), so if you are using the free tier, avoid running it too frequently.

In [12]:
EVAL_CASES = [
    ("I want a charging cable under 500 rupees", "search_products"),
    ("عاوز سماعات بلوتوث", "search_products"),
    ("Compare products 1512 and 1660", "compare_products"),
    ("Is product 1512 available in quantity 3?", "check_stock"),
    ("Add product 1512 to my cart", "add_to_cart"),
    ("What is in my cart?", "view_cart"),
    ("What is the weather today?", None), 
]

passed = 0
for question, expected in EVAL_CASES:
    tid = str(uuid.uuid4())
    result = chat(question, thread_id=tid)
    ok = (expected in result["tools"]) if expected else (not result["tools"])
    passed += ok
    print(f"{'PASS' if ok else 'FAIL'}  [{result['model']}] expected={expected} used={result['tools']} | {question}")
    orders.clear_cart(tid)

print(f"\nTool-selection accuracy: {passed}/{len(EVAL_CASES)}")

PASS  [groq-fast] expected=search_products used=['search_products'] | I want a charging cable under 500 rupees
PASS  [groq-fast] expected=search_products used=['search_products'] | عاوز سماعات بلوتوث
PASS  [groq-smart] expected=compare_products used=['compare_products'] | Compare products 1512 and 1660
PASS  [groq-smart] expected=check_stock used=['check_stock'] | Is product 1512 available in quantity 3?
PASS  [groq-smart] expected=add_to_cart used=['check_stock', 'add_to_cart'] | Add product 1512 to my cart
PASS  [groq-smart] expected=view_cart used=['view_cart'] | What is in my cart?
PASS  [groq-fast] expected=None used=[] | What is the weather today?

Tool-selection accuracy: 7/7


## 12. Fallback test

In [13]:
print("All cloud models skipped -> the local model must answer:")
show(chat("Show me HDMI cables", "fallback-1", skip={"groq-fast", "groq-smart"}))

All cloud models skipped -> the local model must answer:
[simple -> ollama | 113.9s | tools: ['search_products']]
I couldn't find any HDMI cables with the provided keyword. Could you please provide a shorter English keyword or try another category?


## 13. Interactive Chat 
Type `exit` to quit, and `new` for a new conversation (new session).

In [ ]:
thread = str(uuid.uuid4())
while True:
    text = input("\nYou: ").strip()
    if text.lower() == "exit":
        break
    if text.lower() == "new":
        thread = str(uuid.uuid4())
        print("(new conversation, new cart)")
        continue
    if text:
        try:
            show(chat(text, thread))
        except RuntimeError as exc:
            print(exc)

[complex -> groq-smart | 9.3s | tools: ['search_products', 'search_products', 'search_products']]
أعتذر، لا توجد هواتف محمولة في قاعدة البيانات الحالية. هل ترغب في مشاهدة ملحقات الهواتف (مثل الحوامل، الكابلات أو الأغطية) أم تفضل تحديد نوع أو ميزانية معينة للبحث مرة أخرى؟ يرجى إخباري بما تفضله حتى أتمكن من مساعدتك بشكل أفضل.
[simple -> groq-fast | 7.2s | tools: ['search_products']]
إليك بعض خيارات غطاء شاحن متوفرة:

| ID | اسم مختصر | السعر (₹) | التقييم |
|----|-----------|-----------|---------|
| 1893 | LAPSTER Spiral Charger Cable Protector | 99.0 | 4.1 |
| 1732 | Sounce Spiral Charger Cable Protector | 99.0 | 4.0 |

أيّهما تفضّل؟ أو هل تريد إضافة كميّة معينة؟
[warn] groq-smart failed -> trying the next model
[warn] groq-fast failed -> trying the next model
[complex -> ollama | 184.2s | tools: []]
إليك الغطاء الشاحني الأول:

| ID | اسم مختصر | السعر (₹) | التقييم |
|----|-----------|-----------|---------|
| 1893 | LAPSTER Spiral Charger Cable Protector | 99.0 | 4.1 |

هل هذا الغطاء ا

## قفل الـ Checkpoint

```
git status          # make sure .env, data/ and invoices/ are NOT listed
git add notebook2.ipynb orders.py test_orders.py requirements.txt .gitignore
git commit -m "CP4: cart, orders and invoices with confirmation guard"
git tag cp4
git push origin main --tags
```